# Phase 2 — Collinearity diagnostic

Notebook 04 produced a wrong-sign, insignificant coefficient on `yoy_real_EUR_TRY` (β = −5.16, p = 0.13) and a right-sign, significant coefficient on `yoy_real_GBP_TRY` (β = +6.95, p = 0.04). statsmodels flagged a **condition number ≈ 2,560**, which strongly suggests the two FX series are collinear and OLS is splitting the elasticity between them in a numerically unstable way.

This notebook isolates the problem and re-fits three alternative specs:

1. **GBP-only** — drop EUR.
2. **EUR-only** — drop GBP.
3. **Average real-FX index** — replace EUR+GBP with their mean.

Comparing the coefficient on the FX regressor across the four specs (including the no-FE base from notebook 04) tells us whether the EUR/GBP elasticity is a stable economic signal or an artefact of which collinear regressor happens to be in the model.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from pathlib import Path

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'master_monthly.csv',
                 index_col=0, parse_dates=True)
df.index.name = 'date'

def yoy(s): return np.log(s) - np.log(s.shift(12))

df['yoy_arrivals']     = yoy(df['arrivals_total'])
df['yoy_real_EUR_TRY'] = yoy(df['real_EUR_TRY'])
df['yoy_real_GBP_TRY'] = yoy(df['real_GBP_TRY'])
df['yoy_real_USD_TRY'] = yoy(df['real_USD_TRY'])
df['trends_DE_lag1']   = df['trends_DE_Türkei_Urlaub'].shift(1)
df['trends_GB_lag1']   = df['trends_GB_Turkey_holiday'].shift(1)
df['yoy_real_EUGB_avg'] = (df['yoy_real_EUR_TRY'] + df['yoy_real_GBP_TRY']) / 2

## 1. Pair correlations and VIF

If EUR/GBP move together, their pairwise correlation will be near 1 and their VIFs in any joint regression will be large (rule of thumb: VIF > 10 is severe multicollinearity).

In [2]:
fx_cols = ['yoy_real_EUR_TRY', 'yoy_real_GBP_TRY', 'yoy_real_USD_TRY']
print('Pairwise correlations among YoY real-FX series:')
print(df[fx_cols].corr().round(3).to_string())

Pairwise correlations among YoY real-FX series:
                  yoy_real_EUR_TRY  yoy_real_GBP_TRY  yoy_real_USD_TRY
yoy_real_EUR_TRY             1.000             0.864             0.767
yoy_real_GBP_TRY             0.864             1.000             0.797
yoy_real_USD_TRY             0.767             0.797             1.000


In [3]:
# VIF using the same regressor set as notebook 04's base spec.
regressors_base = [
    'yoy_real_EUR_TRY', 'yoy_real_GBP_TRY',
    'trends_DE_lag1', 'trends_GB_lag1',
    'covid', 'russia_war', 'mideast',
]
sample = df[['yoy_arrivals'] + regressors_base].dropna()
X = sm.add_constant(sample[regressors_base])
vif = pd.DataFrame({
    'regressor': X.columns,
    'VIF':       [round(variance_inflation_factor(X.values, i), 2)
                  for i in range(X.shape[1])],
})
vif

,regressor,VIF
0,const,10.17
1,yoy_real_EUR_TRY,15.02
2,yoy_real_GBP_TRY,12.81
3,trends_DE_lag1,1.75
4,trends_GB_lag1,1.83
5,covid,1.26
6,russia_war,2.13
7,mideast,2.19


## 2. Re-fit under three alternatives

Each spec keeps `trends_DE_lag1`, `trends_GB_lag1`, `covid`, `russia_war`, `mideast` and swaps only the FX regressor(s). Newey–West HAC SEs with 4 lags throughout, same sample window per spec (constrained by FX coverage).

In [4]:
def fit_spec(fx_cols_in_spec, label):
    regs = list(fx_cols_in_spec) + [
        'trends_DE_lag1', 'trends_GB_lag1',
        'covid', 'russia_war', 'mideast',
    ]
    sub = df[['yoy_arrivals'] + regs].dropna()
    X = sm.add_constant(sub[regs])
    y = sub['yoy_arrivals']
    m = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    return label, sub, m

specs = {
    'base (EUR+GBP)': ['yoy_real_EUR_TRY', 'yoy_real_GBP_TRY'],
    'GBP-only':       ['yoy_real_GBP_TRY'],
    'EUR-only':       ['yoy_real_EUR_TRY'],
    'avg(EUR,GBP)':   ['yoy_real_EUGB_avg'],
}
results = {label: fit_spec(cols, label) for label, cols in specs.items()}

rows = []
for label, (_, sub, m) in results.items():
    row = {
        'spec':       label,
        'N':          int(m.nobs),
        'R2':         round(m.rsquared, 3),
        'AdjR2':      round(m.rsquared_adj, 3),
        'CondNo':     round(m.condition_number, 0),
        'DW':         round(durbin_watson(m.resid), 3),
    }
    for fx_col in ['yoy_real_EUR_TRY', 'yoy_real_GBP_TRY', 'yoy_real_EUGB_avg']:
        if fx_col in m.params.index:
            row[f'b_{fx_col}'] = round(float(m.params[fx_col]), 2)
            row[f'p_{fx_col}'] = round(float(m.pvalues[fx_col]), 3)
        else:
            row[f'b_{fx_col}'] = np.nan
            row[f'p_{fx_col}'] = np.nan
    rows.append(row)
comparison = pd.DataFrame(rows).set_index('spec')
comparison

,N,R2,AdjR2,CondNo,DW,b_yoy_real_EUR_TRY,p_yoy_real_EUR_TRY,b_yoy_real_GBP_TRY,p_yoy_real_GBP_TRY,b_yoy_real_EUGB_avg,p_yoy_real_EUGB_avg
spec,,,,,,,,,,,
base (EUR+GBP),99,0.218,0.157,2556.0,0.529,-5.16,0.134,6.95,0.040,NaN,NaN
GBP-only,99,0.193,0.141,589.0,0.496,NaN,NaN,2.39,0.054,NaN,NaN
EUR-only,99,0.167,0.113,631.0,0.470,1.84,0.130,NaN,NaN,NaN,NaN
"avg(EUR,GBP)",99,0.181,0.127,618.0,0.483,NaN,NaN,NaN,NaN,2.2,0.081


## 3. Full OLS summaries — single-FX and average-FX specs

The summaries below are the two specs that drop the collinear pair: GBP-only and EUR-only. Compare each coefficient's sign and 95% interval against the joint base spec to see which side the elasticity "belongs" to.

In [5]:
for label in ['GBP-only', 'EUR-only', 'avg(EUR,GBP)']:
    _, sub, m = results[label]
    print(f'=== {label}  N={int(m.nobs)}  R²={m.rsquared:.3f}  AdjR²={m.rsquared_adj:.3f} ===')
    print(m.summary().tables[1])
    print()

=== GBP-only  N=99  R²=0.193  AdjR²=0.141 ===
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.1971      0.195      1.011      0.312      -0.185       0.579
yoy_real_GBP_TRY     2.3867      1.238      1.928      0.054      -0.039       4.812
trends_DE_lag1       0.0016      0.006      0.255      0.798      -0.011       0.014
trends_GB_lag1      -0.0056      0.007     -0.753      0.452      -0.020       0.009
covid               -1.2171      0.926     -1.314      0.189      -3.033       0.599
russia_war           0.3945      0.197      1.999      0.046       0.008       0.781
mideast             -0.1859      0.177     -1.049      0.294      -0.533       0.161

=== EUR-only  N=99  R²=0.167  AdjR²=0.113 ===
                       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------

## 4. Verdict

Cell below picks the spec with the lowest condition number and prints a plain-English read on whether the EUR/GBP collinearity was masking a real elasticity or whether the FX channel is genuinely weak at this aggregation.

In [6]:
alpha = 0.05
best = comparison.sort_values('CondNo').index[0]
best_label, (_, _, best_model) = best, results[best]
_, _, best_model = results[best_label]

fx_in_best = [c for c in ['yoy_real_EUR_TRY', 'yoy_real_GBP_TRY', 'yoy_real_EUGB_avg']
              if c in best_model.params.index]
fx = fx_in_best[0]
b, p = best_model.params[fx], best_model.pvalues[fx]
ci_lo, ci_hi = best_model.conf_int().loc[fx]

print(f'Lowest-condition spec: {best_label}  (condition number {best_model.condition_number:.0f})')
print(f'FX regressor: {fx}')
print(f'  beta = {b:+.3f}   p = {p:.4f}   95% CI = [{ci_lo:+.3f}, {ci_hi:+.3f}]')
print()
if p < alpha and b > 0:
    print('Reading: once the collinear EUR/GBP pair is replaced with a single')
    print(f'regressor, the FX coefficient is {b:+.2f} and significant at 5%.')
    print(f'A 1% real TRY depreciation is associated with a {b:+.2f}% change in YoY arrivals.')
    print('The wrong-sign EUR result in notebook 04 was a collinearity artefact.')
elif p < alpha and b < 0:
    print(f'Single-FX spec also yields a negative significant FX coefficient ({b:+.2f}).')
    print('Collinearity was not the cause — the elasticity itself is wrong-signed in this sample.')
    print('Likely culprits: aggregate arrivals masks country-level heterogeneity, or omitted')
    print('variables (Russian relocation flows, regional safety perception) dominate the macro signal.')
else:
    print(f'Single-FX spec yields beta = {b:+.2f}, p = {p:.3f} — not significant at 5%.')
    print('Collinearity was contributing to instability, but the underlying FX signal is also')
    print('weak. To pin the elasticity down, the natural next steps are: (a) per-source-country')
    print('panel regression instead of aggregate arrivals, (b) richer dynamic spec (AR / DL terms),')
    print('(c) longer / wider FX measure (e.g., REER basket against a broader peer set).')

Lowest-condition spec: GBP-only  (condition number 589)
FX regressor: yoy_real_GBP_TRY
  beta = +2.387   p = 0.0538   95% CI = [-0.039, +4.812]

Single-FX spec yields beta = +2.39, p = 0.054 — not significant at 5%.
Collinearity was contributing to instability, but the underlying FX signal is also
weak. To pin the elasticity down, the natural next steps are: (a) per-source-country
panel regression instead of aggregate arrivals, (b) richer dynamic spec (AR / DL terms),
(c) longer / wider FX measure (e.g., REER basket against a broader peer set).
